# MovieLens 100K: is K=10 a suitable recommendation-list length?

This notebook reproduces the figures in the dataset exploration and verifies the reported values with assertions. It uses the repository's actual `dataset/ml-100k/ml-100k.inter` and RecBole split implementation, not an approximate random split.

Run all cells in order with a Python kernel containing this repository's RecBole dependencies (the repository `.venv`). Start Jupyter from the repository or from this notebook's directory. No model training, downloads, or test-set model evaluation are needed. Outputs are retained for review.

The original analysis used seed 42, per-user random splits, full-catalog candidates, and **all observed ratings as positives**. Both 80/10/10 and the proposed 70/15/15 are examined. These are fixed reproduction settings, not an endorsement of a finalized protocol. The YAML files are loaded as in the project adapter; relevant analysis settings are explicitly fixed and checked below.

MovieLens 100K source: [GroupLens](https://grouplens.org/datasets/movielens/100k/). Local file counts are authoritative for this notebook.

In [1]:
from pathlib import Path
import hashlib
import platform
import sys
import numpy as np
import pandas as pd

ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / 'dataset/ml-100k/ml-100k.inter').is_file()
             and (p / 'recbole').is_dir()), None)
assert ROOT is not None, 'Run from the repository or one of its subdirectories.'
sys.path.insert(0, str(ROOT))
from recbole.config import Config
from recbole.data import create_dataset
from recbole.utils import init_seed
import torch

INTER = ROOT / 'dataset/ml-100k/ml-100k.inter'
print('Python:', platform.python_version())
print('NumPy:', np.__version__, '| pandas:', pd.__version__, '| torch:', torch.__version__)
for relative in ['dataset/ml-100k/ml-100k.inter', 'project/configs/protocol.yaml',
                 'project/configs/models/bpr.yaml', 'recbole/data/dataset/dataset.py']:
    path = ROOT / relative
    print(relative, 'SHA256:', hashlib.sha256(path.read_bytes()).hexdigest())

Python: 3.11.16
NumPy: 2.4.6 | pandas: 3.0.6 | torch: 2.13.0+rocm7.2
dataset/ml-100k/ml-100k.inter SHA256: 4edb74e2a81178c2ba9ff381495f754f996c4aea351b1272ca36b43da0935eff
project/configs/protocol.yaml SHA256: 19e8bff6579bfc227b95f82097f015923561e66fac23f380e84d89c5e19871cd
project/configs/models/bpr.yaml SHA256: 15efb38ca264c136785f749ff4e2e138f92d467e13ff396da7be2750ec4f5b3e
recbole/data/dataset/dataset.py SHA256: 0e8c17f6e33d3a8dc775121d7523af18264f385357f5c3fd571e70f0b560a6dd


## 1. Raw interactions and user activity

Counts refer to distinct raw user/movie IDs, without RecBole's reserved padding ID. Quantiles use NumPy's default linear interpolation. Density is observed pairs divided by all possible user–movie pairs.

In [2]:
ratings = pd.read_csv(INTER, sep='\t')
assert list(ratings.columns) == ['user_id:token', 'item_id:token', 'rating:float', 'timestamp:float']
ratings.columns = ['user', 'item', 'rating', 'timestamp']
user_counts = ratings.groupby('user').size()
item_counts = ratings.groupby('item').size()

def describe(values):
    return {'min': np.min(values), 'p10': np.quantile(values, .1),
            'p25': np.quantile(values, .25), 'median': np.median(values),
            'p75': np.quantile(values, .75), 'p90': np.quantile(values, .9),
            'max': np.max(values), 'mean': np.mean(values)}

raw = {'interactions': len(ratings), 'users': len(user_counts), 'movies': len(item_counts),
       'duplicate_user_movie_pairs': int(ratings.duplicated(['user', 'item']).sum()),
       'missing_cells': int(ratings.isna().sum().sum()),
       'density': len(ratings) / (len(user_counts) * len(item_counts))}
print(pd.Series(raw).to_string())
print('\nActivity counts:')
print(pd.DataFrame({'per_user': describe(user_counts), 'per_movie': describe(item_counts)}).to_string())
print('\nRating distribution:')
print(ratings.rating.value_counts().sort_index().to_string())
print('Movies with <=5 ratings:', int((item_counts <= 5).sum()))
print('Movies with <=10 ratings:', int((item_counts <= 10).sum()))

assert [raw[k] for k in ['interactions', 'users', 'movies', 'duplicate_user_movie_pairs', 'missing_cells']] == [100000, 943, 1682, 0, 0]
assert [user_counts.min(), user_counts.quantile(.25), user_counts.median(),
        user_counts.quantile(.75), user_counts.max()] == [20, 33, 65, 148, 737]
assert round(user_counts.mean()) == 106

interactions                  100000.000000
users                            943.000000
movies                          1682.000000
duplicate_user_movie_pairs         0.000000
missing_cells                      0.000000
density                            0.063047

Activity counts:
          per_user   per_movie
min      20.000000    1.000000
p10      23.000000    2.000000
p25      33.000000    6.000000
median   65.000000   27.000000
p75     148.000000   80.000000
p90     244.400000  169.000000
max     737.000000  583.000000
mean    106.044539   59.453032

Rating distribution:
rating
1     6110
2    11370
3    27145
4    34174
5    21201
Movies with <=5 ratings: 384
Movies with <=10 ratings: 563


## 2. Exact RecBole splits and candidate counts

RecBole floors validation and test allocations per user and assigns the remainder to training. It also shuffles before splitting. `Dataset.build()` reproduces this without training or writing run artifacts.

Candidate policy for these calculations: the entire 1,682-movie catalog, excluding training history for validation and training plus validation history for test. Padding ID 0 is excluded. Cold training items remain eligible; their counts are shown. Candidate policy must be kept consistent across models.

In [3]:
splits = {}
# Config reads sys.argv; suppress Jupyter's kernel-launch arguments temporarily.
original_argv = sys.argv[:]
try:
    sys.argv = [original_argv[0]]
    for label, ratios in [('80/10/10', [.8, .1, .1]), ('70/15/15', [.7, .15, .15])]:
        config = Config(
            config_file_list=[str(ROOT / 'project/configs/models/bpr.yaml'),
                              str(ROOT / 'project/configs/protocol.yaml')],
            config_dict={'data_path': str(ROOT / 'dataset'), 'dataset': 'ml-100k',
                         'seed': 42, 'reproducibility': True, 'use_gpu': False,
                         'save_dataset': False, 'save_dataloaders': False,
                         'eval_args': {'split': {'RS': ratios}, 'group_by': 'user',
                                       'order': 'RO', 'mode': 'full'}})
        for field in ['val_interval', 'user_inter_num_interval', 'item_inter_num_interval', 'threshold']:
            assert config[field] is None, f'Unexpected filtering/relevance setting: {field}'
        init_seed(config['seed'], config['reproducibility'])
        dataset = create_dataset(config)
        train, valid, test = dataset.build()
        assert dataset.user_num - 1 == raw['users']
        assert dataset.item_num - 1 == raw['movies']
        counts = [np.bincount(part.inter_feat['user_id'].numpy(),
                             minlength=dataset.user_num)[1:] for part in (train, valid, test)]
        np.testing.assert_array_equal(counts[1], counts[2])
        for part, expected_fraction in [(1, ratios[1]), (2, ratios[2])]:
            np.testing.assert_array_equal(np.sort(counts[part]),
                                          np.sort(np.floor(user_counts.to_numpy() * expected_fraction)))
        histories = []
        for part in (train, valid, test):
            mapping = {}
            for u, i in zip(part.inter_feat['user_id'].tolist(), part.inter_feat['item_id'].tolist()):
                mapping.setdefault(u, set()).add(i)
            histories.append(mapping)
        for u in range(1, dataset.user_num):
            assert all(histories[a][u].isdisjoint(histories[b][u])
                       for a, b in [(0, 1), (0, 2), (1, 2)])
        splits[label] = {'dataset': dataset, 'parts': (train, valid, test),
                         'counts': counts, 'histories': histories,
                         'valid_candidates': raw['movies'] - counts[0],
                         'test_candidates': raw['movies'] - counts[0] - counts[1]}
finally:
    sys.argv = original_argv

split_table = {}
for label, s in splits.items():
    train_counts, heldout, test_counts = s['counts']
    split_table[label] = {'train_interactions': len(s['parts'][0]),
                         'validation_interactions': len(s['parts'][1]),
                         'test_interactions': len(s['parts'][2]),
                         'minimum_heldout_per_user': heldout.min(),
                         'p25_heldout_per_user': np.quantile(heldout, .25),
                         'median_heldout_per_user': np.median(heldout),
                         'p75_heldout_per_user': np.quantile(heldout, .75),
                         'maximum_heldout_per_user': heldout.max(),
                         'mean_heldout_per_user': heldout.mean(),
                         'users_below_10': int((heldout < 10).sum()),
                         'users_below_10_percent': 100 * (heldout < 10).mean(),
                         'users_above_10': int((heldout > 10).sum()),
                         'users_above_10_percent': 100 * (heldout > 10).mean()}
print(pd.DataFrame(split_table).round(4).to_string())
for label, s in splits.items():
    print('\n' + label + ' candidate counts:')
    print(pd.DataFrame({p: describe(s[p + '_candidates']) for p in ['valid', 'test']}).round(4).to_string())
    trained_items = set(s['parts'][0].inter_feat['item_id'].tolist())
    cold = set(range(1, s['dataset'].item_num)) - trained_items
    print('Movies absent from training:', len(cold))
    print('Validation interactions with those movies:',
          sum(i in cold for i in s['parts'][1].inter_feat['item_id'].tolist()))

for label, expected in [('80/10/10', [80808, 9596, 9596, 2, 3, 6, 14, 73, 579, 325]),
                        ('70/15/15', [70846, 14577, 14577, 3, 4, 9, 22, 110, 483, 433])]:
    row = split_table[label]
    keys = ['train_interactions', 'validation_interactions', 'test_interactions',
            'minimum_heldout_per_user', 'p25_heldout_per_user', 'median_heldout_per_user',
            'p75_heldout_per_user', 'maximum_heldout_per_user', 'users_below_10', 'users_above_10']
    assert [row[k] for k in keys] == expected
assert [round(split_table[x]['users_below_10_percent'], 1) for x in splits] == [61.4, 51.2]
assert [round(split_table[x]['users_above_10_percent'], 1) for x in splits] == [34.5, 45.9]
assert splits['80/10/10']['valid_candidates'].min() == 1091
assert splits['80/10/10']['valid_candidates'].max() == 1666

/tmp/recbole-movielens-notebook/recbole/data/dataset/dataset.py:648: ChainedAssignmentError: A value is being set on a copy of a DataFrame or Series through chained assignment using an inplace method.
Such inplace method never works to update the original DataFrame or Series, because the intermediate object on which we are setting values always behaves as a copy (due to Copy-on-Write).

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' instead, to perform the operation inplace on the original object, or try to avoid an inplace operation using 'df[col] = df[col].method(value)'.

See the documentation for a more detailed explanation: https://pandas.pydata.org/pandas-docs/stable/user_guide/copy_on_write.html
  feat[field].fillna(value=0, inplace=True)
/tmp/recbole-movielens-notebook/recbole/data/dataset/dataset.py:650: ChainedAssignmentError: A value is being set on a copy of a DataFrame or Series through chained assignment us

## 3. Metric ceilings and random-ranking expectations

For a user with `r > 0` held-out positives and a length-`k` list, the most possible hits are `min(k, r)`. Thus maximum recall is `min(k,r)/r`, and maximum precision is `min(k,r)/k`. Macro ceilings average these user-specific maxima equally across users. They assume every held-out positive is eligible, as verified above.

For uniform random selection without replacement from `c` candidates, expected hits are `k*r/c`, expected recall is `k/c`, and expected precision is `r/c`. These are analytic expectations, not a random simulation. All candidate pools here exceed 20.

These ceilings describe the observed labels, not actual user satisfaction. An unobserved rating does not establish that the user dislikes a movie.

In [4]:
ceiling_rows = []
for label, s in splits.items():
    relevant_counts = s['counts'][1]
    candidates = s['valid_candidates']
    assert np.all(relevant_counts > 0) and np.all(candidates >= 20)
    for k in [5, 10, 20]:
        ceiling_rows.append({'split': label, 'K': k,
                             'users_below_K': int((relevant_counts < k).sum()),
                             'users_above_K': int((relevant_counts > k).sum()),
                             'max_macro_recall': np.minimum(1, k / relevant_counts).mean(),
                             'max_macro_precision': np.minimum(1, relevant_counts / k).mean(),
                             'random_expected_recall': (k / candidates).mean(),
                             'random_expected_precision': (relevant_counts / candidates).mean()})
ceilings = pd.DataFrame(ceiling_rows).set_index(['split', 'K'])
print(ceilings.round(6).to_string())
assert round(ceilings.loc[('80/10/10', 10), 'max_macro_recall'], 3) == .846
assert round(ceilings.loc[('80/10/10', 10), 'max_macro_precision'], 3) == .631
assert min(10, 2) / 2 == 1 and min(10, 2) / 10 == .2
assert min(10, 20) / 10 == 1 and min(10, 20) / 20 == .5
print('\nExample: 2 positives => max Recall@10=1, max Precision@10=0.2')
print('Example: 20 positives => max Recall@10=0.5, max Precision@10=1')

             users_below_K  users_above_K  max_macro_recall  max_macro_precision  random_expected_recall  random_expected_precision
split    K                                                                                                                         
80/10/10 5             375            497          0.689796             0.815058                0.003141                   0.006751
         10            579            325          0.846063             0.630859                0.006282                   0.006751
         20            794            135          0.958815             0.436532                0.012565                   0.006751
70/15/15 5             246            645          0.573560             0.914740                0.003118                   0.010096
         10            483            433          0.754057             0.747190                0.006237                   0.010096
         20            673            255          0.899978             0.55

## 4. Optional relevance sensitivity: ratings of at least 4

This changes only the labels within the existing validation splits; it does not filter and re-split the dataset. RecBole may normalize rating values, so the threshold is applied to the original raw ratings after translating both IDs back. Empty relevance sets would require an explicit evaluation policy. The primary reported results continue to use all observed interactions.

In [5]:
raw_ratings = {(str(u), str(i)): r for u, i, r in zip(ratings.user, ratings.item, ratings.rating)}
high_rating_rows = {}
for label, s in splits.items():
    dataset = s['dataset']
    valid = s['parts'][1].inter_feat
    raw_users = dataset.id2token(dataset.uid_field, valid['user_id'].numpy())
    raw_items = dataset.id2token(dataset.iid_field, valid['item_id'].numpy())
    high = np.array([raw_ratings[(u, i)] >= 4 for u, i in zip(raw_users, raw_items)])
    counts = np.bincount(valid['user_id'].numpy()[high], minlength=dataset.user_num)[1:]
    high_rating_rows[label] = {**describe(counts), 'empty_users': int((counts == 0).sum()),
                               'users_below_10': int((counts < 10).sum())}
print(pd.DataFrame(high_rating_rows).to_string())
assert high_rating_rows['80/10/10']['empty_users'] == 63
assert high_rating_rows['70/15/15']['empty_users'] == 23

                  80/10/10    70/15/15
min               0.000000    0.000000
p10               1.000000    1.000000
p25               2.000000    3.000000
median            4.000000    6.000000
p75               8.000000   12.000000
p90              14.000000   20.000000
max              37.000000   54.000000
mean              5.644751    8.513256
empty_users      63.000000   23.000000
users_below_10  756.000000  641.000000


## 5. Training-only popularity baseline, evaluated on validation

Count training interactions for each movie. Rank by descending count; break ties by ascending **RecBole internal item ID**, matching the original analysis. Exclude each user's training items, then retain the first 20 candidates. Cold items remain at the bottom of the catalog ranking.

Precision = hits/K; recall = hits/number of validation positives. MRR is the reciprocal position of the first hit, or zero if none. NDCG uses binary gains and `log2(position+1)` discount, divided by the ideal gain for `min(K, positives)` hits. Each metric is averaged equally across all 943 users. Test labels are never used by this baseline.

The baseline illustrates the cutoff tradeoff; it does not establish an optimal cutoff for hybrid models or rerankers.

In [6]:
def evaluate_ranking(ranking, relevant, k):
    top = ranking[:k]
    assert len(top) == k and len(set(top)) == k and relevant
    hits = np.array([item in relevant for item in top], dtype=float)
    hit_positions = np.flatnonzero(hits)
    dcg = (hits / np.log2(np.arange(k) + 2)).sum()
    ideal_dcg = (1 / np.log2(np.arange(min(k, len(relevant))) + 2)).sum()
    return [hits.sum() / len(relevant), hits.mean(),
            1 / (hit_positions[0] + 1) if len(hit_positions) else 0,
            dcg / ideal_dcg]

# Known-answer examples check averaging inputs and position discounts.
np.testing.assert_allclose(evaluate_ranking([1, 2], {1, 2}, 2), [1, 1, 1, 1])
np.testing.assert_allclose(evaluate_ranking([3, 4], {1, 2}, 2), [0, 0, 0, 0])
np.testing.assert_allclose(evaluate_ranking([3, 1], {1}, 2), [1, .5, .5, 1 / np.log2(3)])

baseline_rows = []
for label, s in splits.items():
    dataset = s['dataset']
    train, valid, _ = s['parts']
    train_history, relevant, _ = s['histories']
    popularity = np.bincount(train.inter_feat['item_id'].numpy(), minlength=dataset.item_num)
    catalog_order = np.lexsort((np.arange(dataset.item_num), -popularity))
    catalog_order = catalog_order[catalog_order != 0]
    scores = {k: [] for k in [5, 10, 20]}
    for user in range(1, dataset.user_num):
        ranking = [int(i) for i in catalog_order if i not in train_history[user]][:20]
        assert set(ranking).isdisjoint(train_history[user])
        for k in scores:
            scores[k].append(evaluate_ranking(ranking, relevant[user], k))
    for k, values in scores.items():
        means = np.mean(values, axis=0)
        baseline_rows.append({'split': label, 'K': k,
                              **dict(zip(['Recall', 'Precision', 'MRR', 'NDCG'], means))})
baseline = pd.DataFrame(baseline_rows).set_index(['split', 'K'])
print(baseline.round(6).to_string())
expected_rounded = np.array([[.072, .099, .115], [.115, .090, .124], [.173, .075, .139]])
np.testing.assert_allclose(baseline.loc['80/10/10', ['Recall', 'Precision', 'NDCG']].round(3),
                           expected_rounded, rtol=0, atol=1e-12)

               Recall  Precision       MRR      NDCG
split    K                                          
80/10/10 5   0.072065   0.099258  0.220042  0.114677
         10  0.115187   0.089608  0.241510  0.124127
         20  0.172723   0.075133  0.250013  0.139107
70/15/15 5   0.066199   0.136373  0.284818  0.147528
         10  0.109978   0.126087  0.308560  0.152942
         20  0.162955   0.103181  0.315706  0.159887


## 6. Interpretation

- **K=10 is a reasonable primary cutoff**, with K=5 and K=20 as sensitivity checks. The catalog easily supports all three lengths.
- Under 80/10/10 the median user has six held-out positives and 61.4% have fewer than ten. Low Precision@10 can partly reflect limited labels, even under perfect ranking.
- Under 70/15/15 the median held-out count rises to nine and validation gains 4,981 interactions (51.9%), while training loses 9,962 interactions. This supplies richer evaluation evidence at the cost of training data; it does not guarantee better model quality or statistical significance.
- Popularity-baseline recall increases with list length and precision decreases in these runs. Dataset counts and one baseline cannot prove K=10 is optimal or that users prefer ten recommendations.
- Recall and NDCG capture aspects of the whole list that MRR misses after the first hit. This matters when comparing diversification, calibration, and fairness rerankers.
- Freeze the protocol before comparing models. Model/settings selection uses validation; test is reserved for final reporting. The descriptive test partition sizes above are not model-performance measurements.

The notebook reproduces the reported descriptive and baseline numbers; the choice of list length remains a project decision.

In [7]:
print('All assertions passed: dataset counts, exact split counts, candidate bounds, metric ceilings, and reported baseline values reproduced.')

All assertions passed: dataset counts, exact split counts, candidate bounds, metric ceilings, and reported baseline values reproduced.
